# Практические задачи: очистка и подготовка данных
Задачи к лекции 5 «Очистка и подготовка данных» (`5.da.fillna.rus.pdf`): пропущенные значения, `dropna` и `fillna`, дубликаты, типы данных, выбросы, очистка строк и проверка согласованности.

Сложность: ⭐ — базовая, ⭐⭐ — средняя, ⭐⭐⭐ — повышенная.

## 0. Данные для практики

Во всех заданиях (если не сказано иное) используется таблица заказов интернет-магазина `df_raw`.

| Столбец | Значение |
| :--- | :--- |
| `order_id` | Номер заказа (должен быть уникальным) |
| `mijoz` | Имя клиента |
| `telefon` | Телефон клиента |
| `shahar` | Город доставки |
| `kategoriya` | Категория товара |
| `narx` | Цена за единицу, сум |
| `soni` | Количество |
| `summa` | Сумма заказа (`narx × soni`) |
| `buyurtma_sana` | Дата заказа |
| `yetkazish_sana` | Дата доставки |

Таблица специально «испорчена»: в ней есть почти все проблемы, разобранные на лекции.

Запустите ячейку ниже. **Каждое задание начинайте с `df = df_raw.copy()`**, если в нём не сказано работать с результатом предыдущего задания.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})

df = df_raw.copy()   # работаем с копией, df_raw не изменяем
df

,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana
0,1001,Алишер Каримов,+998 90 123-45-67,Ташкент,Продукты,12 000,2.0,24000.0,2026-09-01,2026-09-02
1,1002,дилноза юсупова,998911234567,ташкент,Электроника,450000,1.0,450000.0,2026-09-01,2026-09-04
2,1003,Бобур Азимов,90 555 44 33,Самарканд,Продукты,8000,NaN,NaN,2026-09-02,2026-09-03
3,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05
4,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05
5,1005,None,+998(94)7778899,Tashkent,Продукты,?,3.0,NaN,2026-09-03,2026-09-04
6,1006,Жасур Тошев,,г. Самарканд,Электроника,520 000 сум,1.0,520000.0,2026-09-03,2026-09-02
7,1007,Нодира Исмоилова,97 000 11 22,N/A,Одежда,120000,-2.0,-240000.0,2026-09-04,2026-09-06
8,1008,Шахзод Ахмедов,12-34,Бухара,Продукты,1500000,1.0,1500000.0,2026-09-04,2026-09-05
9,1009,Малика Саидова,+998 99 888 77 66,Самарканд,Одежда,75000,2.0,150000.0,2026-09-05,None


## 1. Поиск пропущенных значений

**Задача 1.1** ⭐ Выведите информацию о таблице с помощью `info()`. В каких столбцах есть пропуски по мнению pandas?

**Задача 1.2** ⭐ Посчитайте количество пропусков в каждом столбце и долю пропусков в процентах (округлите до 1 знака). Выведите результат одной таблицей с двумя столбцами: `kol_vo` и `dolya_%`.

**Задача 1.3** ⭐ Выведите все строки, в которых есть хотя бы один пропуск. Сколько их?

**Задача 1.4** ⭐⭐ Найдите **скрытые пропуски**. Выведите `value_counts(dropna=False)` для столбцов `shahar` и `narx`. Какие значения на самом деле означают «нет данных»?

**Задача 1.5** ⭐⭐ Замените все скрытые пропуски (`'N/A'`, `'-'`, `'?'`, `''`) на `np.nan`. Сравните количество пропусков **до** и **после** замены в одной таблице.

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15 non-null     int64  
 1   mijoz           14 non-null     object 
 2   telefon         15 non-null     object 
 3   shahar          15 non-null     object 
 4   kategoriya      15 non-null     object 
 5   narx            15 non-null     object 
 6   soni            13 non-null     float64
 7   summa           11 non-null     float64
 8   buyurtma_sana   15 non-null     object 
 9   yetkazish_sana  13 non-null     object 
dtypes: float64(2), int64(1), object(7)
memory usage: 1.3+ KB


In [ ]:
kol_vo = df.isna().sum()
dolya_pct = (df.isna().mean() * 100).round(1)

missing_info = pd.DataFrame({
    'kol_vo': kol_vo,
    'dolya_%': dolya_pct
})

display(missing_info)

,kol_vo,dolya_%
order_id,0,0.0
mijoz,1,6.7
telefon,0,0.0
shahar,0,0.0
kategoriya,0,0.0
narx,0,0.0
soni,2,13.3
summa,4,26.7
buyurtma_sana,0,0.0
yetkazish_sana,2,13.3


In [ ]:
missing_rows = df[df.isna().any(axis=1)]

display(missing_rows)
print(f"Количество строк с пропусками: {len(missing_rows)}")

,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana
2,1003,Бобур Азимов,90 555 44 33,Самарканд,Продукты,8000,NaN,NaN,2026-09-02,2026-09-03
5,1005,None,+998(94)7778899,Tashkent,Продукты,?,3.0,NaN,2026-09-03,2026-09-04
9,1009,Малика Саидова,+998 99 888 77 66,Самарканд,Одежда,75000,2.0,150000.0,2026-09-05,None
13,1012,Зарина Холова,95-333-44-55,-,Одежда,110000,NaN,NaN,нет даты,None
14,1013,Отабек Юлдашев,+998901112233,Наманган,Электроника,,2.0,NaN,2026-09-07,2026-09-09


Количество строк с пропусками: 5


In [ ]:
(df['shahar'].value_counts(dropna=False))
(df['narx'].value_counts(dropna=False))

,count
narx,
95000,2
75000,2
12 000,1
8000,1
450000,1
?,1
520 000 сум,1
120000,1
1500000,1


In [ ]:
before = df.isna().sum()
df_replaced = df.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
after = df_replaced.isna().sum()
comparison_df = pd.DataFrame({
    'propuski_do': before,
    'propuski_posle': after
})

display(comparison_df)

,propuski_do,propuski_posle
order_id,0,0
mijoz,1,1
telefon,0,1
shahar,0,2
kategoriya,0,0
narx,0,2
soni,2,2
summa,4,4
buyurtma_sana,0,0
yetkazish_sana,2,2


## 2. Стратегии работы с пропусками: dropna и fillna

Перед заданиями выполните замену скрытых пропусков из задачи 1.5:
```python
df = df_raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
```

**Задача 2.1** ⭐ Сколько строк останется после каждого вызова? Выведите `shape` для:
* `df.dropna()`
* `df.dropna(subset=['soni'])`
* `df.dropna(how='all')`
* `df.dropna(thresh=9)`

Какой вариант потерял больше всего данных и почему?

**Задача 2.2** ⭐ Заполните пропуски в столбце `mijoz` значением `'Неизвестно'`, а в столбце `shahar` — **модой** (самым частым городом).

**Задача 2.3** ⭐⭐ Для столбца `soni` посчитайте среднее и медиану. Заполните пропуски медианой. Затем сравните среднее значение `soni` в трёх вариантах: пропуски удалены, пропуски заполнены нулём, пропуски заполнены медианой. Почему заполнение нулём — плохая идея для количества?

**Задача 2.4** ⭐⭐ Перед заполнением `soni` создайте столбец-флаг `soni_nan`, который показывает, было ли значение пропущено. Выведите строки, где флаг равен `True`.

**Задача 2.5** ⭐⭐ Дан ряд температур за неделю:
```python
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range('2026-09-01', periods=7))
```
Заполните пропуски тремя способами: `ffill()`, `bfill()` и `interpolate()`. Выведите результаты рядом в одной таблице. Почему последнее значение после `bfill()` осталось пустым?

**Задача 2.6** ⭐⭐⭐ Дана таблица товаров:
```python
tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
```
Заполните пропущенные цены **медианой своей категории** (через `groupby(...).transform('median')`). Сравните с заполнением общей медианой. Какой вариант разумнее?

In [ ]:
print(len(df.dropna()))
print(len(df.dropna(subset=['soni'])))
print(len(df.dropna(how='all')))
print(len(df.dropna(thresh=9)))

10
13
15
12


In [ ]:
import numpy as np
import pandas as pd

df = df_raw.replace({
    'N/A': np.nan,
    '-': np.nan,
    '?': np.nan,
    '': np.nan
})

df['mijoz'] = df['mijoz'].fillna('Неизвестно')

moda = df['shahar'].mode()[0]
df['shahar'] = df['shahar'].fillna(moda)

print("Самый частый город:", moda)
print(df[['mijoz', 'shahar']])

Самый частый город: Бухара
                mijoz        shahar
0     Алишер Каримов        Ташкент
1     дилноза юсупова      ташкент 
2        Бобур Азимов     Самарканд
3     Камола Рахимова        Бухара
4     Камола Рахимова        Бухара
5          Неизвестно      Tashkent
6         Жасур Тошев  г. Самарканд
7    Нодира Исмоилова        Бухара
8      Шахзод Ахмедов        Бухара
9      Малика Саидова     Самарканд
10     Малика Саидова     Самарканд
11     алишер каримов       ТАШКЕНТ
12      Сардор Нуриев      Наманган
13      Зарина Холова        Бухара
14     Отабек Юлдашев      Наманган


In [ ]:
print(df['soni'].mean())
print(df['soni'].median())

mediana = df['soni'].median()

print(df['soni'].dropna().mean())
print(df['soni'].fillna(0).mean())
print(df['soni'].fillna(mediana).mean())

df['soni'] = df['soni'].fillna(mediana)


1.4615384615384615
1.0
1.4615384615384615
1.2666666666666666
1.4


In [ ]:
df['soni_nan'] = df['soni'].isna()

df[df['soni_nan'] == True]

,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana,soni_nan


In [ ]:
temp = pd.Series(
    [18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
    index=pd.date_range('2026-09-01', periods=7)
)

result = pd.DataFrame({
    'ffill': temp.ffill(),
    'bfill': temp.bfill(),
    'interpolate': temp.interpolate()
})

result

,ffill,bfill,interpolate
2026-09-01,18.0,18.0,18.0
2026-09-02,18.0,22.0,20.0
2026-09-03,22.0,22.0,22.0
2026-09-04,22.0,28.0,24.0
2026-09-05,22.0,28.0,26.0
2026-09-06,28.0,28.0,28.0
2026-09-07,28.0,NaN,28.0


In [ ]:
import pandas as pd
import numpy as np

tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})

tovar['narx_median_category'] = tovar['narx'].fillna(
    tovar.groupby('kategoriya')['narx'].transform('median')
)

tovar['narx_median_all'] = tovar['narx'].fillna(tovar['narx'].median())

tovar

,kategoriya,narx,narx_median_category,narx_median_all
0,Продукты,5000.0,5000.0,5000.0
1,Продукты,NaN,7000.0,80000.0
2,Продукты,9000.0,9000.0,9000.0
3,Одежда,80000.0,80000.0,80000.0
4,Одежда,NaN,100000.0,80000.0
5,Одежда,120000.0,120000.0,120000.0
6,Электроника,400000.0,400000.0,400000.0
7,Электроника,NaN,400000.0,80000.0


## 3. Дублирующиеся строки

**Задача 3.1** ⭐ Сколько в таблице `df_raw` полностью одинаковых строк? Выведите **все** копии (включая первые вхождения).

**Задача 3.2** ⭐ Проверьте, уникален ли столбец `order_id`. Сколько номеров заказов повторяется?

**Задача 3.3** ⭐⭐ Заказ `1009` встречается дважды, но строки не полностью одинаковые. Чем они отличаются? Удалите дубликаты по `order_id` так, чтобы осталась **последняя** (самая свежая) запись. Перенумеруйте индекс. Сколько строк осталось?

**Задача 3.4** ⭐⭐⭐ Клиенты `'  Алишер Каримов '` и `'алишер каримов'` — один и тот же человек. Сколько **уникальных** клиентов в столбце `mijoz` до и после очистки имён (`strip` + `title`)? Выведите список клиентов, которые сделали больше одного заказа (после удаления дубликатов заказов из задачи 3.3).

In [ ]:
duplicates = df_raw[df_raw.duplicated(keep=False)]

print("Количество полностью одинаковых строк:", duplicates.shape[0])

duplicates


Количество полностью одинаковых строк: 2


,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana
3,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05
4,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05


In [ ]:
print("Уникален:", df_raw['order_id'].is_unique)

print("Количество повторяющихся номеров:",
      df_raw['order_id'].duplicated().sum())

df_raw[df_raw['order_id'].duplicated(keep=False)].sort_values('order_id')

Уникален: False
Количество повторяющихся номеров: 2


,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana
3,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05
4,1004,Камола Рахимова,93-111-22-33,Бухара,Одежда,95000,1.0,95000.0,2026-09-02,2026-09-05
9,1009,Малика Саидова,+998 99 888 77 66,Самарканд,Одежда,75000,2.0,150000.0,2026-09-05,None
10,1009,Малика Саидова,+998 99 888 77 66,Самарканд,Одежда,75000,2.0,150000.0,2026-09-05,2026-09-07


In [ ]:
print(df_raw[df_raw['order_id'] == 1009].T)

df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)

print(df)
print("Строк осталось:", len(df))

                               9                  10
order_id                     1009               1009
mijoz              Малика Саидова     Малика Саидова
telefon         +998 99 888 77 66  +998 99 888 77 66
shahar                  Самарканд          Самарканд
kategoriya                 Одежда             Одежда
narx                        75000              75000
soni                          2.0                2.0
summa                    150000.0           150000.0
buyurtma_sana          2026-09-05         2026-09-05
yetkazish_sana               None         2026-09-07
    order_id              mijoz            telefon        shahar   kategoriya  \
0       1001    Алишер Каримов   +998 90 123-45-67       Ташкент     Продукты   
1       1002    дилноза юсупова       998911234567      ташкент   Электроника   
2       1003       Бобур Азимов       90 555 44 33     Самарканд     Продукты   
3       1004    Камола Рахимова       93-111-22-33        Бухара       Одежда   
4       1005

In [ ]:
print("До очистки:", df['mijoz'].nunique())

df['mijoz'] = df['mijoz'].str.strip().str.title()

print("После очистки:", df['mijoz'].nunique())

print(df['mijoz'].value_counts()[df['mijoz'].value_counts() > 1])

До очистки: 12
После очистки: 11
mijoz
Алишер Каримов    2
Name: count, dtype: int64


## 4. Исправление типов данных

**Задача 4.1** ⭐ Выведите типы всех столбцов `df_raw`. Какие столбцы имеют неправильный тип и какой тип должен быть у них?

**Задача 4.2** ⭐ Попробуйте выполнить `df_raw['narx'].astype(int)`. Какая ошибка возникла и почему? (Оберните код в `try / except`, чтобы ноутбук не остановился.)

**Задача 4.3** ⭐⭐ Преобразуйте `narx` в число с помощью `pd.to_numeric(..., errors='coerce')` **без предварительной очистки**. Выведите исходные значения, которые превратились в `NaN`. Какие из них были на самом деле корректными ценами?

**Задача 4.4** ⭐⭐ Теперь сначала очистите `narx`: удалите все символы, кроме цифр (подсказка: `.str.replace(r'\D', '', regex=True)`), затем преобразуйте в число. Сколько значений теперь стало `NaN`?

**Задача 4.5** ⭐⭐ Преобразуйте `buyurtma_sana` и `yetkazish_sana` в даты, **явно указав формат** `'%Y-%m-%d'`. Какие значения не удалось распознать? Выведите минимальную и максимальную дату заказа.

**Задача 4.6** ⭐⭐ Столбец `soni` имеет тип `float64`, хотя количество — целое число. Преобразуйте его в тип `Int64`, сохранив пропуски.

**Задача 4.7** ⭐⭐ Преобразуйте `kategoriya` в тип `category`. Сравните объём памяти до и после для ряда `pd.Series(df_raw['kategoriya'].tolist() * 10000)` (используйте `memory_usage(deep=True)`).

**Задача 4.8** ⭐ Дан ряд ответов на вопрос «Нужна ли доставка?»:
```python
otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])
```
Преобразуйте его в `True` / `False`.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})

df = df_raw.copy()
df = df.replace({
    'N/A': np.nan,
    '-': np.nan,
    '?': np.nan,
    '': np.nan
})
df.dtypes

,0
order_id,int64
mijoz,object
telefon,object
shahar,object
kategoriya,object
narx,object
soni,float64
summa,float64
buyurtma_sana,object
yetkazish_sana,object


In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})

try:
    df_raw['narx'].astype(int)
except Exception as e:
    print(f"Тип ошибки: {type(e).__name__}")
    print(f"Текст ошибки: {e}")

Тип ошибки: ValueError
Текст ошибки: invalid literal for int() with base 10: '12 000'


In [ ]:
import pandas as pd
import numpy as np

# 1. Применяем pd.to_numeric с errors='coerce' без предварительной очистки
narx_numeric = pd.to_numeric(df_raw['narx'], errors='coerce')

# 2. Вытаскиваем только те исходные значения, которые превратились в NaN
original_nans = df_raw.loc[narx_numeric.isna(), 'narx']

print("Исходные значения из столбца 'narx', которые превратились в NaN:")
display(original_nans)

Исходные значения из столбца 'narx', которые превратились в NaN:


,narx
0,12 000
5,?
6,520 000 сум
12,6 500
14,


In [ ]:

cleaned_narx = df_raw['narx'].str.replace(r'\D', '', regex=True)

narx_numeric_cleaned = pd.to_numeric(cleaned_narx, errors='coerce')

nan_count = narx_numeric_cleaned.isna().sum()

print(f"Количество значений NaN после очистки и преобразования: {nan_count}")
print("\nРезультаты преобразования:")
display(pd.DataFrame({
    'Исходное значение': df_raw['narx'],
    'Очищенное числовое': narx_numeric_cleaned
}))

Количество значений NaN после очистки и преобразования: 2

Результаты преобразования:


,Исходное значение,Очищенное числовое
0,12 000,12000.0
1,450000,450000.0
2,8000,8000.0
3,95000,95000.0
4,95000,95000.0
5,?,NaN
6,520 000 сум,520000.0
7,120000,120000.0
8,1500000,1500000.0
9,75000,75000.0


In [ ]:
buyurtma_dt = pd.to_datetime(df_raw['buyurtma_sana'], format='%Y-%m-%d', errors='coerce')
yetkazish_dt = pd.to_datetime(df_raw['yetkazish_sana'], format='%Y-%m-%d', errors='coerce')

unresolved_buyurtma = df_raw.loc[buyurtma_dt.isna() & df_raw['buyurtma_sana'].notna(), 'buyurtma_sana'].unique()
unresolved_yetkazish = df_raw.loc[yetkazish_dt.isna() & df_raw['yetkazish_sana'].notna(), 'yetkazish_sana'].unique()

print("Не удалось распознать в 'buyurtma_sana':", unresolved_buyurtma)
print("Не удалось распознать в 'yetkazish_sana':", unresolved_yetkazish)

min_date = buyurtma_dt.min()
max_date = buyurtma_dt.max()
print(f"Минимальная дата заказа: {min_date.strftime('%Y-%m-%d') if pd.notna(min_date) else 'NaT'}")
print(f"Максимальная дата заказа: {max_date.strftime('%Y-%m-%d') if pd.notna(max_date) else 'NaT'}")

Не удалось распознать в 'buyurtma_sana': ['нет даты']
Не удалось распознать в 'yetkazish_sana': []
Минимальная дата заказа: 2026-09-01
Максимальная дата заказа: 2026-09-07


In [ ]:

df = df_raw.copy()

# Преобразуем столбец 'soni' в тип 'Int64' (сохраняет пропуски как <NA>)
df['soni'] = df['soni'].astype('Int64')

# Выведем типы данных и обновленный столбец для наглядности
print("Тип столбца 'soni' теперь:", df['soni'].dtype)
display(df[['order_id', 'soni']])

Тип столбца 'soni' теперь: Int64


,order_id,soni
0,1001,2
1,1002,1
2,1003,<NA>
3,1004,1
4,1004,1
5,1005,3
6,1006,1
7,1007,-2
8,1008,1
9,1009,2


In [ ]:


large_series_object = pd.Series(df_raw['kategoriya'].tolist() * 10000)


large_series_category = large_series_object.astype('category')

mem_object = large_series_object.memory_usage(deep=True)
mem_category = large_series_category.memory_usage(deep=True)

print(f"Объем памяти (тип object): {mem_object:,} байт")
print(f"Объем памяти (тип category): {mem_category:,} байт")
print(f"Оптимизация: память уменьшилась в {mem_object / mem_category:.1f} раз(а)!")

Объем памяти (тип object): 14,850,132 байт
Объем памяти (тип category): 150,541 байт
Оптимизация: память уменьшилась в 98.6 раз(а)!


In [ ]:

otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])

# Приводим к нижнему регистру и убираем лишние пробелы
cleaned_otvet = otvet.str.strip().str.lower()

# Преобразуем в True / False с помощью словаря соответствий
mapping = {
    'ha': True,
    'yes': True,
    'yo\'q': False,
    'no': False
}

boolean_otvet = cleaned_otvet.map(mapping)

# Выведем результат
display(pd.DataFrame({
    'Исходный ответ': otvet,
    'Преобразованный': boolean_otvet
}))

,Исходный ответ,Преобразованный
0,ha,True
1,Yo'q,False
2,HA,True
3,yes,True
4,no,False
5,Ha,True


## 5. Выбросы (outliers)

Для заданий этого раздела используйте очищенную цену:
```python
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()
```

**Задача 5.1** ⭐ Выведите `describe()` для `narx`. Сравните среднее и медиану. Что подсказывает разница между `75%` и `max`?

**Задача 5.2** ⭐⭐ Напишите функцию `iqr_bounds(s, k=1.5)`, которая возвращает нижнюю и верхнюю границы по методу IQR. Найдите выбросы в `narx`.

**Задача 5.3** ⭐⭐ Найдите выбросы в `narx` по z-оценке (`|z| > 3`) и по робастной z-оценке (через медиану и MAD, порог 3.5). Выведите максимальное значение `|z|` для обоих методов. Насколько «уверенно» каждый метод нашёл выброс? Объясните разницу.

**Задача 5.4** ⭐⭐⭐ Цены сильно зависят от категории: 1 500 000 — нормально для электроники, но не для продуктов. Найдите выбросы по IQR **внутри каждой категории** (подсказка: `groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.25))`). Какой заказ оказался выбросом?

**Задача 5.5** ⭐⭐ Даны баллы за экзамен (максимум — 100):
```python
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
```
Найдите невозможные значения по **бизнес-правилу** (балл от 0 до 100). Затем ограничьте значения диапазоном `[0, 100]` с помощью `clip`. Подумайте: является ли балл `0` ошибкой?

In [ ]:
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()

print("Описательные статистики столбца 'narx':")
display(narx.describe())

print(f"Среднее значение: {narx.mean():,.2f}")
print(f"Медиана: {narx.median():,.2f}")

Описательные статистики столбца 'narx':


,narx
count,1.300000e+01
mean,2.651154e+05
std,4.092300e+05
min,6.500000e+03
25%,7.500000e+04
50%,9.500000e+04
75%,3.800000e+05
max,1.500000e+06


Среднее значение: 265,115.38
Медиана: 95,000.00


In [ ]:
def iqr_bounds(s, k=1.5):
    q25 = s.quantile(0.25)
    q75 = s.quantile(0.75)
    iqr = q75 - q25
    lower_bound = q25 - k * iqr
    upper_bound = q75 + k * iqr
    return lower_bound, upper_bound

lower, upper = iqr_bounds(narx)
print(f"Границы IQR (k=1.5): [{lower:,.1f}, {upper:,.1f}]")

outliers_iqr = narx[(narx < lower) | (narx > upper)]
print("Выбросы по методу IQR:")
display(outliers_iqr)

Границы IQR (k=1.5): [-382,500.0, 837,500.0]
Выбросы по методу IQR:


,narx
8,1500000.0


In [ ]:
import numpy as np

# Обычная z-оценка
mean_val = narx.mean()
std_val = narx.std()
z_scores = (narx - mean_val) / std_val
outliers_z = narx[z_scores.abs() > 3]

# Робастная z-оценка
median_val = narx.median()
mad_val = (narx - median_val).abs().median()
# Константа 1.4826 для нормализации под стандартное отклонение
robust_z_scores = 0.6745 * (narx - median_val) / mad_val
outliers_robust = narx[robust_z_scores.abs() > 3.5]

print(f"Максимальное значение |z| (обычное): {z_scores.abs().max():.2f}")
print(f"Максимальное значение |robust z| (робастное): {robust_z_scores.abs().max():.2f}")
print("\nВыбросы по обычной z-оценке:")
display(outliers_z)
print("\nВыбросы по робастной z-оценке:")
display(outliers_robust)

Максимальное значение |z| (обычное): 3.02
Максимальное значение |robust z| (робастное): 11.42

Выбросы по обычной z-оценке:


,narx
8,1500000.0



Выбросы по робастной z-оценке:


,narx
8,1500000.0


In [ ]:

df_temp = df_raw.copy()
df_temp['narx_cleaned'] = pd.to_numeric(df_temp['narx'].str.replace(r'\D', '', regex=True), errors='coerce')


q25 = df_temp.groupby('kategoriya')['narx_cleaned'].transform(lambda s: s.quantile(0.25))
q75 = df_temp.groupby('kategoriya')['narx_cleaned'].transform(lambda s: s.quantile(0.75))
iqr = q75 - q25

lower_by_cat = q25 - 1.5 * iqr
upper_by_cat = q75 + 1.5 * iqr

outliers_cat = df_temp[(df_temp['narx_cleaned'] < lower_by_cat) | (df_temp['narx_cleaned'] > upper_by_cat)]
print("Заказы, являющиеся выбросами внутри своей категории:")
display(outliers_cat[['order_id', 'mijoz', 'kategoriya', 'narx']])

Заказы, являющиеся выбросами внутри своей категории:


,order_id,mijoz,kategoriya,narx
8,1008,Шахзод Ахмедов,Продукты,1500000


In [ ]:

ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])


invalid_balls = ball[(ball < 0) | (ball > 100)]
print("Невозможные баллы:", invalid_balls.tolist())


clipped_balls = ball.clip(lower=0, upper=100)
print("Ограниченные баллы:", clipped_balls.tolist())


Невозможные баллы: [150, -5]
Ограниченные баллы: [72, 85, 64, 91, 0, 78, 100, 88, 69, 100, 95, 0]


In [ ]:
# Ваш код здесь


## 6. Очистка строк: strip, replace, regex

**Задача 6.1** ⭐ Приведите столбец `mijoz` к единому виду: уберите лишние пробелы по краям и внутри строки, каждое слово — с заглавной буквы.

**Задача 6.2** ⭐⭐ Нормализуйте столбец `shahar`. В результате должны остаться только значения `Ташкент`, `Самарканд`, `Бухара`, `Наманган` и `NaN`. Выведите `value_counts(dropna=False)` до и после.
Подсказки: `strip` → `lower` → убрать префикс `'г. '` с помощью regex → словарь замен → `title`.

**Задача 6.3** ⭐⭐ Нормализуйте телефоны к формату `+998XXXXXXXXX`:
1. удалите все символы, кроме цифр;
2. если цифр 12 и номер начинается с `998` — оставьте последние 9 цифр; если цифр 9 — оставьте как есть; иначе — `NaN`;
3. добавьте `'+998'` в начало.

Какие номера оказались некорректными?

**Задача 6.4** ⭐⭐ Дан ряд цен в произвольном формате:
```python
price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', '15000 so\'m', 'договорная', '1 200 000'])
```
Извлеките число из каждой строки с помощью `.str.extract`, очистите разделители и преобразуйте в число.

**Задача 6.5** ⭐⭐ Дан ряд email-адресов:
```python
emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
```
Очистите адреса (пробелы, нижний регистр) и проверьте, соответствует ли каждый адрес формату `имя@домен.зона` с помощью `.str.match`.

**Задача 6.6** ⭐ Дан ряд адресов:
```python
manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
```
Разделите его на три столбца: `shahar`, `tuman`, `uy` с помощью `.str.split(..., expand=True)`.

**Задача 6.7** ⭐⭐⭐ Найдите в `df_raw` всех клиентов, в имени которых есть буквосочетание «ов» (без учёта регистра), и всех клиентов, чьё **имя** (первое слово) начинается на «А» или «Б».

In [ ]:

df_strings = df_raw.copy()
df_strings['mijoz'] = df_strings['mijoz'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
print("Очищенные имена клиентов:")
display(df_strings['mijoz'].unique())

Очищенные имена клиентов:


array(['Алишер Каримов', 'Дилноза Юсупова', 'Бобур Азимов',
       'Камола Рахимова', 'None', 'Жасур Тошев', 'Нодира Исмоилова',
       'Шахзод Ахмедов', 'Малика Саидова', 'Сардор Нуриев',
       'Зарина Холова', 'Отабек Юлдашев'], dtype=object)

In [ ]:
print("Частоты до нормализации:")
display(df_raw['shahar'].value_counts(dropna=False))

shahar_clean = df_raw['shahar'].astype(str).str.strip().str.lower()
shahar_clean = shahar_clean.str.replace(r'^\s*г\.\s*', '', regex=True)
city_mapping = {
    'tashkent': 'Ташкент',
    'ташкент': 'Ташкент',
    'самарканд': 'Самарканд',
    'бухара': 'Бухара',
    'наманган': 'Наманган',
    'nan': np.nan,
    '-': np.nan,
    'n/a': np.nan
}
df_strings['shahar'] = shahar_clean.map(city_mapping)
print("\nЧастоты после нормализации:")
display(df_strings['shahar'].value_counts(dropna=False))

Частоты до нормализации:


,count
shahar,
Бухара,3
Самарканд,3
Наманган,2
Ташкент,1
ташкент,1
Tashkent,1
г. Самарканд,1
N/A,1
ТАШКЕНТ,1



Частоты после нормализации:


,count
shahar,
Ташкент,4
Самарканд,4
Бухара,3
NaN,2
Наманган,2


In [ ]:

def normalize_phone(phone):
    if pd.isna(phone):
        return np.nan
    digits = ''.join(filter(str.isdigit, str(phone)))
    if len(digits) == 12 and digits.startswith('998'):
        return '+' + digits
    elif len(digits) == 9:
        return '+998' + digits
    else:
        return np.nan

df_strings['telefon_normalized'] = df_raw['telefon'].apply(normalize_phone)
print("Сравнение исходных и нормализованных телефонов:")
display(pd.DataFrame({
    'Исходный': df_raw['telefon'],
    'Нормализованный': df_strings['telefon_normalized']
}))

Сравнение исходных и нормализованных телефонов:


,Исходный,Нормализованный
0,+998 90 123-45-67,+998901234567
1,998911234567,+998911234567
2,90 555 44 33,+998905554433
3,93-111-22-33,+998931112233
4,93-111-22-33,+998931112233
5,+998(94)7778899,+998947778899
6,,NaN
7,97 000 11 22,+998970001122
8,12-34,NaN
9,+998 99 888 77 66,+998998887766


In [ ]:

price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', '15000 so\'m', 'договорная', '1 200 000'])
extracted_price = price_txt.str.replace(r'\s+', '', regex=True).str.extract(r'(\d+[.,]?\d*)')[0]
numeric_price = pd.to_numeric(extracted_price.str.replace(',', ''), errors='coerce')
display(pd.DataFrame({
    'Текст': price_txt,
    'Результат': numeric_price
}))

,Текст,Результат
0,12 000 сум,12000.0
1,цена: 9500,9500.0
2,"7,500",7500.0
3,15000 so'm,15000.0
4,договорная,NaN
5,1 200 000,1200000.0


In [ ]:

emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
cleaned_emails = emails.str.strip().str.lower()
pattern = r'^[a-zA-Z0-9_.-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,4}$'
matches = cleaned_emails.str.match(pattern)
display(pd.DataFrame({
    'Email': emails,
    'Валидный?': matches
}))

,Email,Валидный?
0,ali@mail.uz,True
1,Bobur@Gmail.com,True
2,kamola.mail.ru,False
3,test@,False
4,sardor_n@tiu.uz,True


In [ ]:

manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
split_manzil = manzil.str.split(', ', expand=True)
split_manzil.columns = ['shahar', 'tuman', 'uy']
display(split_manzil)

,shahar,tuman,uy
0,Ташкент,Чиланзар,12
1,Самарканд,Регистан,5
2,Бухара,Центр,7


In [ ]:

cleaned_names = df_raw['mijoz'].fillna('').str.strip().str.title()
mask_ov = cleaned_names.str.contains('ов', case=False)
mask_ab = cleaned_names.str.contains(r'^[АБ]', case=True)
print("Клиенты с 'ов':")
display(df_raw[mask_ov]['mijoz'].unique())
print("\nКлиенты, имя которых начинается на А или Б:")
display(df_raw[mask_ab]['mijoz'].unique())

Клиенты с 'ов':


array(['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов',
       'Камола Рахимова', 'Нодира Исмоилова', 'Шахзод Ахмедов',
       'Малика Саидова', 'алишер каримов', 'Зарина Холова'], dtype=object)


Клиенты, имя которых начинается на А или Б:


array(['  Алишер Каримов ', 'Бобур Азимов', 'алишер каримов'],
      dtype=object)

## 7. Проверка согласованности данных

Для заданий этого раздела подготовьте таблицу:
```python
df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
```

**Задача 7.1** ⭐ Найдите заказы с неположительным количеством (`soni <= 0`).

**Задача 7.2** ⭐ Найдите заказы, у которых дата доставки **раньше** даты заказа.

**Задача 7.3** ⭐⭐ Проверьте правило `summa == narx × soni`. Выведите строки, где правило нарушено (строки с пропусками в этих столбцах не учитывайте).

**Задача 7.4** ⭐⭐ Допустимые категории: `Продукты`, `Одежда`, `Электроника`, `Бытовая техника`. Проверьте, что все категории допустимы.

**Задача 7.5** ⭐⭐⭐ Напишите функцию `check(df)`, которая возвращает `pd.Series` с количеством проблем по каждой проверке:
* повторяющиеся `order_id`;
* `soni <= 0`;
* `narx <= 0`;
* `summa` не равна `narx × soni`;
* доставка раньше заказа;
* дата заказа в будущем;
* общее количество пропусков.

Запустите её на `df`.

**Задача 7.6** ⭐⭐ Добавьте проверки `assert`: номера заказов уникальны, количество положительно. Убедитесь, что на неочищенных данных `assert` срабатывает (оберните в `try / except AssertionError`).

In [ ]:

df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

invalid_quantity = df[df['soni'] <= 0]
print("Заказы с неположительным количеством (soni <= 0):")
display(invalid_quantity[['order_id', 'mijoz', 'soni']])

Заказы с неположительным количеством (soni <= 0):


,order_id,mijoz,soni
6,1007,Нодира Исмоилова,-2.0


In [ ]:

# Поиск заказов, у которых дата доставки раньше даты заказа
delivery_earlier = df[df['yetkazish_sana'] < df['buyurtma_sana']]
print("Заказы с доставкой раньше даты заказа:")
display(delivery_earlier[['order_id', 'mijoz', 'buyurtma_sana', 'yetkazish_sana']])

Заказы с доставкой раньше даты заказа:


,order_id,mijoz,buyurtma_sana,yetkazish_sana
5,1006,Жасур Тошев,2026-09-03,2026-09-02


In [ ]:

# Проверка правила summa == narx * soni
# Исключаем строки с пропусками в этих столбцах
mask_valid = df['summa'].notna() & df['narx'].notna() & df['soni'].notna()
calculated_sum = df['narx'] * df['soni']
summa_mismatch = df[mask_valid & (df['summa'] != calculated_sum)]

print("Заказы с расхождением в сумме (summa != narx * soni):")
display(summa_mismatch[['order_id', 'mijoz', 'narx', 'soni', 'summa']])

Заказы с расхождением в сумме (summa != narx * soni):


,order_id,mijoz,narx,soni,summa
9,1010,алишер каримов,380000.0,1.0,400000.0


In [ ]:

# Проверка допустимости категорий
allowed_categories = {'Продукты', 'Одежда', 'Электроника', 'Бытовая техника'}
invalid_categories = df[~df['kategoriya'].isin(allowed_categories)]

print(f"Допустимые категории: {allowed_categories}")
print("Заказы с недопустимыми категориями:")
display(invalid_categories[['order_id', 'kategoriya']])

Допустимые категории: {'Электроника', 'Одежда', 'Продукты', 'Бытовая техника'}
Заказы с недопустимыми категориями:


,order_id,kategoriya


In [ ]:

# Функция check(df), собирающая все проблемы в один pd.Series
def check(dataframe):
    issues = {}
    # 1. Повторяющиеся order_id
    issues['повторяющиеся_order_id'] = dataframe['order_id'].duplicated().sum()

    # Безопасно преобразуем soni и narx в числа для проведения проверок
    temp_soni = pd.to_numeric(dataframe['soni'], errors='coerce')
    try:
        temp_narx = pd.to_numeric(dataframe['narx'].astype(str).str.replace(r'\D', '', regex=True), errors='coerce')
    except:
        temp_narx = pd.to_numeric(dataframe['narx'], errors='coerce')

    # 2. soni <= 0
    issues['soni <= 0'] = (temp_soni <= 0).sum()
    # 3. narx <= 0
    issues['narx <= 0'] = (temp_narx <= 0).sum()
    # 4. summa != narx * soni
    temp_summa = pd.to_numeric(dataframe['summa'], errors='coerce')
    mask_val = temp_summa.notna() & temp_narx.notna() & temp_soni.notna()
    issues['ошибка_расчета_summa'] = (mask_val & (temp_summa != temp_narx * temp_soni)).sum()

    # 5. доставка раньше заказа
    temp_buyurtma = pd.to_datetime(dataframe['buyurtma_sana'], errors='coerce')
    temp_yetkazish = pd.to_datetime(dataframe['yetkazish_sana'], errors='coerce')
    issues['доставка_раньше_заказа'] = (temp_yetkazish < temp_buyurtma).sum()

    # 6. дата заказа в будущем (считаем будущее относительно 2026-10-01)
    issues['дата_заказа_в_будущем'] = (temp_buyurtma > pd.to_datetime('2026-10-01')).sum()
    # 7. общее количество пропусков
    issues['общее_число_пропусков'] = dataframe.isna().sum().sum()

    return pd.Series(issues)

print("Анализ качества данных на текущем датасете:")
display(check(df))

Анализ качества данных на текущем датасете:


,0
повторяющиеся_order_id,0
soni <= 0,1
narx <= 0,0
ошибка_расчета_summa,9
доставка_раньше_заказа,1
дата_заказа_в_будущем,0
общее_число_пропусков,11


In [ ]:

# Проверки assert на неочищенных данных
try:
    # На исходных неочищенных данных df_raw:
    print("Проверяем уникальность order_id в df_raw...")
    assert df_raw['order_id'].is_unique, "order_id содержит дубликаты!"
except AssertionError as e:
    print(f"Перехвачена ошибка AssertionError: {e}")

try:
    print("Проверяем, что количество всегда больше нуля в df_raw...")
    assert (df_raw['soni'] > 0).all() or df_raw['soni'].isna().any(), "Присутствуют неположительные значения soni!"
    assert (df_raw['soni'].dropna() > 0).all(), "Обнаружено количество <= 0!"
except AssertionError as e:
    print(f"Перехвачена ошибка AssertionError: {e}")

Проверяем уникальность order_id в df_raw...
Перехвачена ошибка AssertionError: order_id содержит дубликаты!
Проверяем, что количество всегда больше нуля в df_raw...
Перехвачена ошибка AssertionError: Обнаружено количество <= 0!


## 8. Итоговое задание: конвейер очистки

**Задача 8.1** ⭐⭐⭐ Напишите функцию `clean(raw)`, которая возвращает очищенную копию `df_raw`. Соблюдайте порядок шагов из лекции:

1. **Скрытые пропуски** → `NaN`.
2. **Строки**: `mijoz` (пробелы, регистр), `shahar` (единое написание), `telefon` (формат `+998XXXXXXXXX`).
3. **Типы**: `narx` → число, даты → `datetime`, `kategoriya` → `category`.
4. **Дубликаты**: по `order_id`, оставить последнюю запись.
5. **Ошибки и выбросы**:
   * `soni <= 0` → `NaN`;
   * выбросы цены по IQR **внутри категории** → `NaN`;
   * дата доставки раньше даты заказа → `NaN` в `yetkazish_sana`.
6. **Пропуски**:
   * `narx` — медиана категории;
   * `soni` — медиана, затем тип `int`;
   * `mijoz`, `shahar` — `'Неизвестно'`;
   * `buyurtma_sana` — предыдущая дата (`ffill`); `yetkazish_sana` оставить пустой (заказ мог быть ещё не доставлен).
7. **Пересчитать** `summa = narx × soni`.

**Задача 8.2** ⭐⭐ Запустите `check()` из задачи 7.5 **до** и **после** очистки и выведите результаты рядом.

**Задача 8.3** ⭐⭐ Добавьте в конце `assert`-проверки: `order_id` уникален, `soni > 0`, в `narx` и `summa` нет пропусков.

**Задача 8.4** ⭐⭐ Составьте таблицу «было — стало»: количество строк, пропусков, уникальных городов, уникальных клиентов, средняя и медианная сумма заказа.

**Задача 8.5** ⭐⭐⭐ (Для размышления) Заказ `1008`: продукт за 1 500 000. Мы заменили цену медианой категории. Предложите другой способ обработки и объясните, когда он был бы лучше.

In [ ]:

def clean(raw):
    # Создаем копию
    df_clean = raw.copy()

    # 1. Скрытые пропуски -> NaN
    df_clean = df_clean.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})

    # 2. Строки: mijoz (пробелы, регистр)
    df_clean['mijoz'] = df_clean['mijoz'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
    df_clean.loc[df_clean['mijoz'] == 'None', 'mijoz'] = np.nan

    # Строки: shahar (единое написание)
    shahar_clean = df_clean['shahar'].astype(str).str.strip().str.lower()
    shahar_clean = shahar_clean.str.replace(r'^\s*г\.\s*', '', regex=True)
    city_mapping = {
        'tashkent': 'Ташкент',
        'ташкент': 'Ташкент',
        'самарканд': 'Самарканд',
        'бухара': 'Бухара',
        'наманган': 'Наманган'
    }
    df_clean['shahar'] = shahar_clean.map(city_mapping)

    # Строки: telefon (формат +998XXXXXXXXX)
    def clean_phone(phone):
        if pd.isna(phone):
            return np.nan
        digits = ''.join(filter(str.isdigit, str(phone)))
        if len(digits) == 12 and digits.startswith('998'):
            return '+' + digits
        elif len(digits) == 9:
            return '+998' + digits
        else:
            return np.nan
    df_clean['telefon'] = df_clean['telefon'].apply(clean_phone)

    # 3. Типы: narx -> число, даты -> datetime
    df_clean['narx'] = pd.to_numeric(df_clean['narx'].astype(str).str.replace(r'\D', '', regex=True), errors='coerce')
    for col in ['buyurtma_sana', 'yetkazish_sana']:
        df_clean[col] = pd.to_datetime(df_clean[col], format='%Y-%m-%d', errors='coerce')

    # 4. Дубликаты: по order_id, оставить последнюю запись
    df_clean = df_clean.drop_duplicates(subset='order_id', keep='last')

    # 5. Ошибки и выбросы
    # soni <= 0 -> NaN
    df_clean.loc[df_clean['soni'] <= 0, 'soni'] = np.nan

    # Выбросы цены по IQR внутри категории -> NaN
    q25 = df_clean.groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.25))
    q75 = df_clean.groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.75))
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr
    df_clean.loc[(df_clean['narx'] < lower_bound) | (df_clean['narx'] > upper_bound), 'narx'] = np.nan

    # Дата доставки раньше даты заказа -> NaN в yetkazish_sana
    df_clean.loc[df_clean['yetkazish_sana'] < df_clean['buyurtma_sana'], 'yetkazish_sana'] = pd.NaT

    # 6. Пропуски
    # narx — медиана категории
    df_clean['narx'] = df_clean['narx'].fillna(df_clean.groupby('kategoriya')['narx'].transform('median'))
    # soni — медиана, сначала приведем к float, заполним медианой, округлим и безопасно приведем к Int64
    df_clean['soni'] = pd.to_numeric(df_clean['soni'], errors='coerce')
    soni_median = df_clean['soni'].median()
    df_clean['soni'] = df_clean['soni'].fillna(soni_median).round().astype('Int64')

    # mijoz, shahar — 'Неизвестно'
    df_clean['mijoz'] = df_clean['mijoz'].fillna('Неизвестно')
    df_clean['shahar'] = df_clean['shahar'].fillna('Неизвестно')
    # buyurtma_sana — ffill; yetkazish_sana оставить пустой
    df_clean['buyurtma_sana'] = df_clean['buyurtma_sana'].ffill()

    # Тип категории
    df_clean['kategoriya'] = df_clean['kategoriya'].astype('category')

    # 7. Пересчитать summa = narx * soni
    df_clean['summa'] = df_clean['narx'] * df_clean['soni'].astype(float)

    return df_clean.reset_index(drop=True)

df_cleaned = clean(df_raw)
print("Таблица после работы конвейера очистки:")
display(df_cleaned)

Таблица после работы конвейера очистки:


,order_id,mijoz,telefon,shahar,kategoriya,narx,soni,summa,buyurtma_sana,yetkazish_sana
0,1001,Алишер Каримов,+998901234567,Ташкент,Продукты,12000.0,2,24000.0,2026-09-01,2026-09-02
1,1002,Дилноза Юсупова,+998911234567,Ташкент,Электроника,450000.0,1,450000.0,2026-09-01,2026-09-04
2,1003,Бобур Азимов,+998905554433,Самарканд,Продукты,8000.0,2,16000.0,2026-09-02,2026-09-03
3,1004,Камола Рахимова,+998931112233,Бухара,Одежда,95000.0,1,95000.0,2026-09-02,2026-09-05
4,1005,Неизвестно,+998947778899,Ташкент,Продукты,8000.0,3,24000.0,2026-09-03,2026-09-04
5,1006,Жасур Тошев,NaN,Самарканд,Электроника,520000.0,1,520000.0,2026-09-03,NaT
6,1007,Нодира Исмоилова,+998970001122,Неизвестно,Одежда,120000.0,2,240000.0,2026-09-04,2026-09-06
7,1008,Шахзод Ахмедов,NaN,Бухара,Продукты,8000.0,1,8000.0,2026-09-04,2026-09-05
8,1009,Малика Саидова,+998998887766,Самарканд,Одежда,75000.0,2,150000.0,2026-09-05,2026-09-07
9,1010,Алишер Каримов,+998901234567,Ташкент,Электроника,380000.0,1,380000.0,2026-09-06,2026-09-08


In [ ]:
# Сравнение проверок check() до и после очистки
# Мы воспользуемся функцией check(), реализованной в задаче 7.5
# Если она уже была определена, применим ее к обеим таблицам

try:
    problems_before = check(df_raw)
    problems_after = check(df_cleaned)

    comparison_problems = pd.DataFrame({
        'Было (raw)': problems_before,
        'Стало (cleaned)': problems_after
    })
    print("Сравнение количества проблем в данных:")
    display(comparison_problems)
except NameError:
    # Локальная безопасная версия check на случай отсутствия глобальной функции
    def local_check(dataframe):
        issues = {}
        issues['повторяющиеся_order_id'] = dataframe['order_id'].duplicated().sum()

        temp_soni = pd.to_numeric(dataframe['soni'], errors='coerce')
        try:
            temp_narx = pd.to_numeric(dataframe['narx'].astype(str).str.replace(r'\D', '', regex=True), errors='coerce')
        except:
            temp_narx = pd.to_numeric(dataframe['narx'], errors='coerce')

        issues['soni <= 0'] = (temp_soni <= 0).sum()
        issues['narx <= 0'] = (temp_narx <= 0).sum()

        temp_summa = pd.to_numeric(dataframe['summa'], errors='coerce')
        mask_val = temp_summa.notna() & temp_narx.notna() & temp_soni.notna()
        issues['ошибка_расчета_summa'] = (mask_val & (temp_summa != temp_narx * temp_soni)).sum()
        issues['общее_число_пропусков'] = dataframe.isna().sum().sum()
        return pd.Series(issues)

    comparison_problems = pd.DataFrame({
        'Было (raw)': local_check(df_raw),
        'Стало (cleaned)': local_check(df_cleaned)
    })
    print("Сравнение количества проблем в данных (локальная проверка):")
    display(comparison_problems)

Сравнение количества проблем в данных:


,Было (raw),Стало (cleaned)
повторяющиеся_order_id,2,0
soni <= 0,1,0
narx <= 0,0,0
ошибка_расчета_summa,1,13
доставка_раньше_заказа,1,0
дата_заказа_в_будущем,0,0
общее_число_пропусков,9,4


In [ ]:

# assert проверки в конце скрипта
try:
    # Гарантируем наличие df_cleaned
    if 'df_cleaned' not in locals() and 'clean' in locals():
        df_cleaned = clean(df_raw)

    assert df_cleaned['order_id'].is_unique, "Критическая ошибка: order_id не уникальны!"
    assert (df_cleaned['soni'] > 0).all(), "Критическая ошибка: обнаружено количество <= 0!"
    assert df_cleaned['narx'].notna().all(), "Критическая ошибка: в ценах остались пропуски!"
    assert df_cleaned['summa'].notna().all(), "Критическая ошибка: в суммах остались пропуски!"
    print("Все assert-проверки успешно пройдены!")
except AssertionError as e:
    print(f"Ошибка при финальной проверке: {e}")
except NameError as e:
    print(f"Не найдена функция clean или df_raw: {e}")

Все assert-проверки успешно пройдены!


In [ ]:

# Сравнение показателей «Было — Стало»
if 'df_cleaned' not in locals() and 'clean' in locals():
    df_cleaned = clean(df_raw)

before_stats = {
    'Строк': len(df_raw),
    'Пропусков': df_raw.isna().sum().sum(),
    'Уникальных городов': df_raw['shahar'].nunique(),
    'Уникальных клиентов': df_raw['mijoz'].nunique(),
    'Средняя сумма': pd.to_numeric(df_raw['summa'], errors='coerce').mean(),
    'Медианная сумма': pd.to_numeric(df_raw['summa'], errors='coerce').median()
}

after_stats = {
    'Строк': len(df_cleaned),
    'Пропусков': df_cleaned.isna().sum().sum(),
    'Уникальных городов': df_cleaned['shahar'].nunique(),
    'Уникальных клиентов': df_cleaned['mijoz'].nunique(),
    'Средняя сумма': df_cleaned['summa'].mean(),
    'Медианная сумма': df_cleaned['summa'].median()
}

comparison_table = pd.DataFrame({
    'Было (df_raw)': pd.Series(before_stats),
    'Стало (df_cleaned)': pd.Series(after_stats)
})

print("Таблица показателей До и После очистки данных:")
display(comparison_table)

Таблица показателей До и После очистки данных:


,Было (df_raw),Стало (df_cleaned)
Строк,15.000000,13.000000
Пропусков,9.000000,4.000000
Уникальных городов,10.000000,5.000000
Уникальных клиентов,12.000000,12.000000
Средняя сумма,288181.818182,234846.153846
Медианная сумма,150000.000000,150000.000000


## 9. Вопросы для самопроверки

Ответьте письменно, 1–3 предложения на вопрос.

1. Почему `np.nan == np.nan` возвращает `False`? Как правильно проверять пропуски?
2. Что такое скрытые пропуски? Приведите три примера.
3. Чем отличаются MCAR, MAR и MNAR? Приведите пример MNAR.
4. Почему `df.dropna()` без параметров может удалить большую часть таблицы, даже если в каждом столбце мало пропусков?
5. Когда для заполнения пропусков лучше использовать медиану, а не среднее?
6. Почему в pandas 3 запись `df['col'].fillna(0, inplace=True)` не изменяет таблицу?
7. Почему дубликаты нужно искать **после** очистки строк?
8. В чём опасность `errors='coerce'`? Как её избежать?
9. Почему z-оценка может не найти очевидный выброс?
10. Чем отличается `Series.str.replace()` от `Series.replace()`?
11. Приведите два примера межстолбцовых (логических) проверок для таблицы заказов.
12. Зачем использовать `assert` в конце скрипта очистки?

1. **Почему `np.nan == np.nan` возвращает `False`? Как правильно проверять пропуски?**
   Согласно стандарту IEEE 754, `NaN` обозначает математическую неопределённость, поэтому он не равен ничему, включая самого себя. Для проверки пропусков нужно использовать метод `.isna()` или `pd.isna()`.

2. **Что такое скрытые пропуски? Приведите три примера.**
   Это отсутствующие данные, записанные в виде допустимого строкового значения вместо системного `NaN`. Примеры: `'-'`, `'?'`, `'N/A'` или пустая строка `''`.

3. **Чем отличаются MCAR, MAR и MNAR? Приведите пример MNAR.**
   * **MCAR**: пропуски абсолютно случайны.
   * **MAR**: вероятность пропуска зависит от других известных параметров.
   * **MNAR**: вероятность пропуска зависит от самого скрытого значения (например, люди с очень высоким доходом скрывают его в опросах).

4. **Почему `df.dropna()` без параметров может удалить большую часть таблицы, даже если в каждом столбце мало пропусков?**
   По умолчанию используется режим `how='any'`, удаляющий всю строку целиком, если в ней есть хотя бы один пропуск. Если единичные пропуски распределены по разным строкам, это может привести к удалению почти всех записей таблицы.

5. **Когда для заполнения пропусков лучше использовать медиану, а не среднее?**
   Медиану лучше использовать при наличии сильной асимметрии данных или экстремальных выбросов (например, цены или зарплаты), так как она устойчива к аномалиям.

6. **Почему в pandas 2.0+ / 3.0 запись `df['col'].fillna(0, inplace=True)` не изменяет таблицу?**
   Параметр `inplace=True` признан устаревшим (deprecated), так как он вызывает проблемы со стабильностью памяти и SettingWithCopy; теперь следует использовать явное переприсваивание: `df['col'] = df['col'].fillna(0)`.

7. **Почему дубликаты нужно искать после очистки строк?**
   До очистки строк неявные дубликаты (например, с разным регистром `'Ташкент'` и `'ташкент '`) не будут распознаны как одинаковые из-за различий в форматировании и пробелах.

8. **В чём опасность `errors='coerce'`? Как её избежать?**
   Она молча преобразует любые нераспознанные форматы или опечатки (например, `'12O00'` с буквой O) в `NaN`, приводя к потере данных. Избежать этого можно предварительной regex-очисткой или использованием режима `errors='raise'` для анализа ошибок.

9. **Почему z-оценка может не найти очевидный выброс?**
   Среднее значение и стандартное отклонение, на которых строится z-оценка, сами сильно смещаются под влиянием экстремальных выбросов, искусственно расширяя интервал нормы.

10. **Чем отличается `Series.str.replace()` от `Series.replace()`?**
    `str.replace()` выполняет поиск и частичную замену подстрок внутри текстовых элементов (часто через regex), а `replace()` заменяет значение ячейки целиком.

11. **Приведите два примера межстолбцовых (логических) проверок для таблицы заказов.**
    * Дата доставки должна быть строго позже даты заказа (`yetkazish_sana >= buyurtma_sana`).
    * Итоговая сумма должна равняться произведению цены на количество (`summa == narx * soni`).

12. **Зачем использовать `assert` в конце скрипта очистки?**
    Это автоматический тест, гарантирующий, что итоговый датасет строго соответствует критическим бизнес-требованиям (нет дубликатов, отрицательных значений или пропусков) перед передачей в модель.